# Caso 12 – Clasificación de posible fuga de clientes usando modelos de aprendizaje automático

En este caso trabajaremos con una base de datos de clientes de una empresa de servicios, con el objetivo de predecir qué clientes tienen mayor probabilidad de abandonar el servicio (churn) frente a aquellos que permanecerán activos. El conjunto de datos incluye variables demográficas, de uso del servicio y de comportamiento del cliente. Se aplicarán modelos de clasificación como Regresión Logística, Máquinas de Vectores de Soporte (SVM) y Gradient Boosting, junto con un proceso riguroso de preprocesamiento que incluye codificación de variables y escalamiento. Para evaluar el desempeño de los modelos, se utilizarán métricas como la curva ROC, el F1 Score y la matriz de confusión, permitiendo analizar la precisión, sensibilidad y especificidad de cada enfoque.

## Resultado Previsto de Aprendizaje
Al finalizar este caso, el estudiante será capaz de:

- Aplicar técnicas de preprocesamiento adecuadas para problemas de clasificación binaria.
- Entrenar y evaluar modelos de clasificación supervisados como Logistic Regression, SVM y Gradient Boosting.
- Interpretar métricas como la curva ROC, matriz de confusión y F1 Score para medir el rendimiento de los modelos.

## Contexto del problema

Usted forma parte del equipo de análisis de una empresa que busca reducir la tasa de pérdida de clientes. El objetivo es identificar, mediante técnicas de aprendizaje automático, qué usuarios tienen mayor probabilidad de abandonar el servicio para poder aplicar estrategias preventivas. Mediante el uso de modelos supervisados, se desarrollará una solución capaz de clasificar a los clientes en riesgo, facilitando la toma de decisiones enfocadas en la retención y mejora del servicio.

Iniciemos importando las librerías necesarias

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, f1_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import RepeatedStratifiedKFold
import pandas as pd

Ahora importamos nuestros datos

In [ ]:
df_train = pd.read_csv('data/customer_churn_dataset-training-master.csv')
df_test = pd.read_csv('data/customer_churn_dataset-testing-master.csv')
df = pd.concat([df_train, df_test]).reset_index(drop=True)
df.head(5)

Veamos las estadísticas básicas de los datos numéricos

In [ ]:
df.describe()

Veamos si contamos con valores faltantes

In [ ]:
df.isna().sum()

Podemos eliminar las filas con valores faltantes, la base de datos es robusta, por lo que este cambio no tendrá mayor efecto

In [ ]:
df.dropna(inplace=True)

# EDA - Análisis de tendencias

A continuación haremos algunas gráficas que nos permitan ver cómo se distribuyen los datos y si hay alguna dependencia con el churn o no

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df, x='Age', hue = 'Churn', bins=15, kde=True, stat = 'density')
plt.title('Distribution of Age')
plt.xlabel('Age')
plt.show()

In [ ]:
plt.figure(figsize=(6, 3))
sns.countplot(x='Gender', hue='Churn', data=df)
plt.title('Gender Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(6, 3))
sns.histplot(df, x='Usage Frequency', hue = 'Churn', bins=15, kde=True, stat = 'density')
plt.title('Distribution of Usage Frequency')
plt.xlabel('Usage Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(6, 3))
sns.boxplot(x='Churn', y='Support Calls', data=df)
plt.title('Support Calls vs Churn')
plt.xlabel('Churn (0 = No, 1 = Yes)')
plt.show()

In [ ]:
plt.figure(figsize=(6, 3))
sns.countplot(x='Subscription Type', hue='Churn', data=df)
plt.title('Subscription Type vs Churn')
plt.show()

In [ ]:
plt.figure(figsize=(6, 3))
sns.countplot(x='Contract Length', hue='Churn', data=df)
plt.title('Contract Length vs Churn')
plt.show()

Al parecer, la mayoría de personas con contratos mensuales han hecho churn. Veamos cuántas son:

In [ ]:
df['Contract Length'].value_counts()

In [ ]:
plt.figure(figsize=(6, 3))
sns.boxplot(x='Churn', y='Total Spend', data=df)
plt.title('Total Spend vs Churn')
plt.xlabel('Churn (0 = No, 1 = Yes)')
plt.show()

Este análisis nos permite ver que la mayoría de las variables en nuestra base de datos son importantes para determinar el Churn, ya que se comportan diferente en los casos de churn y en los casos de no churn.

Ahora, daremos una representación numérica a las variables categóricas. Empecemos con el género

In [ ]:
gender_map = {'Male': 0, 'Female': 1}

df['Gender'] = df['Gender'].map(gender_map)


Ahora usaremos el `OneHotEncoder` de Scikit Learn para crear una representación numérica de las variables categóricas restantes:

In [ ]:
# Creamos el objeto OneHotEncoder
enc = OneHotEncoder(drop='first', sparse_output=False)  # drop='first' to avoid dummy variable trap

# Ajustamos el codificador a las columnas categóricas
categorical_cols = ['Subscription Type', 'Contract Length'] # Add other categorical columns here
encoded_categorical = enc.fit_transform(df[categorical_cols]).toarray()

# Convertimos los datos a dataframe
encoded_categorical_df = pd.DataFrame(
    encoded_categorical,
    columns=enc.get_feature_names_out(categorical_cols)  # Get feature names from the encoder
)


# Combinamos los datos numéricos con los categóricos
final_df = pd.concat([df.drop(columns = categorical_cols).reset_index(drop=True), encoded_categorical_df.reset_index(drop=True)], axis=1)
final_df

Como se ve, para cada posible categoría se ha creado una nueva columna, en la columna correspondiente a la categoría real ahora hay un 1, y en el resto de categorías habrá 0's. Estas columnas claramente no son independientes, ya que su suma debe ser igual a 1. Esto puede generar problemas de multicolinearidad.

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(final_df.corr(),  annot=True, cmap='coolwarm', square=True, vmax=1, vmin=-1)
plt.title('Correlation Matrix')
plt.show()

El churn tiene una correlación significativa con muchas de las variables que tenemos en la base de datos. En sintonía con lo que observamos previamente, el tipo de suscripción es la variable que menos correlación muestra.

Por otro lado, se observa una correlación alta entre las columnas correspondientes al One Hot Encoding de cada variable categórica. Es una buena práctica eliminar cualquiera de estas columnas para entrenar un modelo de Machine Learning

### Estandarización

Para el primer algoritmo que exploraremos, la regresión logística, es importante que las variables estén en la misma escala, sobretodo con motivos de interpretación de los resultados. En este caso, entonces, implementaremos una estandarización estándar.

In [ ]:
# Separate features (X) and target variable (y)
X = final_df.drop(['Churn','CustomerID'], axis=1)
y = final_df['Churn']

# Initialize StandardScaler
scaler = StandardScaler()

# Fit and transform the features
X_scaled = scaler.fit_transform(X)

# Convert the scaled features back to a DataFrame (optional but recommended)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X_scaled_df, y, test_size= 0.2 , shuffle=True, random_state=55, stratify=y)

# Regresión Logística: Concepto y Propiedades

La regresión logística es un modelo estadístico ampliamente utilizado en Machine Learning para resolver problemas de **clasificación binaria**. A pesar de su nombre, la regresión logística no se utiliza para problemas de regresión continua, sino para predecir la probabilidad de pertenencia a una de dos clases.

---

## 1. ¿Qué es la Regresión Logística?

La regresión logística modela la relación entre un conjunto de variables independientes (predictoras) $ X $ y una variable dependiente binaria $ y $, que toma valores en $ \{0, 1\} $. En lugar de predecir directamente el valor de $ y $, el modelo calcula la **probabilidad** de que $ y = 1 $ utilizando la función logística (o sigmoide):

$$
P(y=1 | X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_n X_n)}}
$$

Donde:
- $ \beta_0 $ es el término independiente (intercepto).
- $ \beta_1, \beta_2, \dots, \beta_n $ son los coeficientes asociados a las variables independientes.

El modelo predice $ y = 1 $ si la probabilidad $ P(y=1 | X) $ supera un cierto umbral (por defecto 0.5), y $ y = 0 $ en caso contrario.

---

## 2. Principales Propiedades de la Regresión Logística

### a) **Modelo Lineal en el Espacio Logit**

Aunque la salida del modelo es no lineal (probabilidades entre 0 y 1), la regresión logística es lineal en el **espacio logit**. El logit es el logaritmo de las probabilidades (odds) y se define como:

$$
\text{Logit}(P) = \log\left(\frac{P}{1-P}\right) = \beta_0 + \beta_1 X_1 + \dots + \beta_n X_n
$$

Esto significa que la regresión logística asume una relación lineal entre las variables predictoras y el logit de la probabilidad.

---

### b) **Salida Probabilística**

El modelo produce probabilidades en el rango [0, 1], lo que permite interpretar la salida como la **confianza** del modelo en la predicción. Por ejemplo:
- Una probabilidad de 0.9 indica una alta confianza en la clase positiva.
- Una probabilidad de 0.5 indica incertidumbre.

---

### c) **Clasificación Binaria**

Por defecto, la regresión logística se utiliza para problemas de clasificación binaria ($ y \in \{0, 1\} $). Sin embargo, puede extenderse a problemas multiclase utilizando variantes como la **regresión logística multinomial**.

---

### d) **Simplicidad e Interpretabilidad**

La regresión logística es un modelo:
- **Sencillo de entender e interpretar**: Cada coeficiente $ \beta_i $ representa el cambio en el logit de la probabilidad por unidad de cambio en $ X_i $.
- **Rápido de entrenar**: Es eficiente computacionalmente y funciona bien con conjuntos de datos pequeños a medianos.

---

### e) **Capacidad para Manejar Variables Categóricas**

La regresión logística puede manejar variables categóricas mediante técnicas como la **codificación one-hot**, transformando las categorías en variables binarias.

---

### f) **Asume Independencia de Variables**

El modelo asume que las variables independientes son estadísticamente independientes entre sí. Si las variables están altamente correlacionadas, esto puede causar problemas de **multicolinealidad**, afectando la estabilidad de los coeficientes $ \beta_i $.

---

## 3. Ventajas y Limitaciones

### Ventajas:
1. **Fácil de interpretar**: Las probabilidades predichas son intuitivas.
2. **Rápida y eficiente**: Requiere menos recursos computacionales que modelos más complejos.
3. **Robusta con datos pequeños**: Puede funcionar bien con muestras pequeñas si las variables están bien seleccionadas.
4. **Regularización disponible**: Métodos como la regresión logística regularizada (L1 o L2) permiten manejar problemas de sobreajuste.

### Limitaciones:
1. **Relación lineal en el logit**: No captura relaciones complejas entre las variables.
2. **No multiclase por defecto**: Necesita extensiones para problemas con más de dos clases.
3. **Sensible a valores atípicos**: Los datos extremos pueden influir en los coeficientes estimados.
4. **Asume independencia de observaciones**: No maneja bien datos dependientes (por ejemplo, series temporales) sin modificaciones.

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, Y_train)

Evaluemos el modelo gráficamente

In [ ]:
# Evaluate the best model on the test set
y_pred_prob = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(Y_test, y_pred_prob)
fpr, tpr, thresholds = roc_curve(Y_test, y_pred_prob)

print(f"ROC AUC on test set: {roc_auc}")

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()

# Curva ROC: ¿Qué es y cómo se interpreta?

La curva ROC (*Receiver Operating Characteristic*) es una herramienta ampliamente utilizada para evaluar el desempeño de un modelo de clasificación, especialmente en problemas de clasificación binaria. Proporciona una representación gráfica de la capacidad del modelo para distinguir entre clases positivas y negativas.

---

## 1. ¿Qué es la curva ROC?

La curva ROC muestra la relación entre dos métricas clave, calculadas a diferentes umbrales de decisión:

- **True Positive Rate (TPR)** o *Sensibilidad*:
  $$
  \text{TPR} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Negatives (FN)}}
  $$
  Representa la proporción de positivos correctamente identificados por el modelo.

- **False Positive Rate (FPR)**:
  $$
  \text{FPR} = \frac{\text{False Positives (FP)}}{\text{False Positives (FP)} + \text{True Negatives (TN)}}
  $$
  Representa la proporción de negativos incorrectamente clasificados como positivos.

La curva se construye trazando el **TPR** en el eje $y$ y el **FPR** en el eje $x$, a medida que se varía el umbral de decisión del modelo.

---

## 2. Interpretación de la curva ROC

### a) Forma de la curva
- Una **curva más cercana al punto superior izquierdo** (TPR = 1, FPR = 0) indica un mejor desempeño del modelo.
- Una curva diagonal (línea $y = x$) representa un modelo aleatorio, sin capacidad discriminativa.

### b) Área Bajo la Curva (AUC-ROC)
El área bajo la curva (AUC, por sus siglas en inglés) resume el desempeño del modelo en un solo número entre 0 y 1:
- **AUC = 1**: Modelo perfecto (siempre clasifica correctamente).
- **AUC = 0.5**: Modelo sin capacidad predictiva (equivalente a adivinar al azar).
- **AUC < 0.5**: Peor que un modelo aleatorio, posiblemente porque las clases están invertidas.

### c) Comparación entre modelos
- Cuanto mayor sea el AUC, mejor será el modelo en general.
- Si dos modelos tienen curvas similares, evalúa el desempeño en un rango específico de FPR o TPR que sea más relevante para tu problema.

---

## 3. Usos comunes de la curva ROC

1. **Evaluación de modelos binarios**:
   - Compara modelos para elegir el que tenga mejor discriminación.
   - Ayuda a determinar qué tan bien separa el modelo las clases positivas de las negativas.

2. **Elección del umbral óptimo**:
   - Identificar un punto específico en la curva donde el balance entre TPR y FPR sea adecuado para tu problema.
   - Por ejemplo, en problemas médicos podrías preferir maximizar la sensibilidad (TPR) para minimizar los falsos negativos (FN).

---

## 4. Ventajas y limitaciones de la curva ROC

### Ventajas:
- Es independiente de la distribución de clases (no se ve afectada por el desbalance de clases).
- Proporciona una evaluación integral del modelo en múltiples umbrales.

### Limitaciones:
- Menos informativa en escenarios con alta desproporción de clases. En estos casos, la curva PR (*Precision-Recall*) puede ser más útil.
- No considera el costo relativo de los errores (FP vs FN) ni el umbral específico para la decisión.

La curva ROC nos da información sobre qué tan diferentes son las distribuciones de score para los positivos y los negativos.

In [ ]:
# Separate probabilities for positive and negative cases
positive_probs = y_pred_prob[Y_test == 1]
negative_probs = y_pred_prob[Y_test == 0]

# Define the number of bins
num_bins = 50

# Plot the distributions
plt.figure(figsize=(10, 6))
plt.hist(positive_probs, bins=num_bins, alpha=0.5, label='Positive Cases', density=True)
plt.hist(negative_probs, bins=num_bins, alpha=0.5, label='Negative Cases', density=True)
plt.xlabel('Predicted Probability')
plt.ylabel('Density')
plt.title('Distribution of Predicted Probabilities for Positive and Negative Cases')
plt.legend()
plt.show()

Como se observa, la distribución de los puntajes que el modelo asigna a los positivos y a los negativos es significativamente diferente. Esto nos muestra que el modelo tiene la capacidad de discernir entre estas dos poblaciones a partir de la información entregada.

# Métricas de Evaluación para Modelos de Clasificación

Evaluar correctamente un modelo de clasificación es fundamental para asegurarse de que funciona bien y cumple con los objetivos del problema. Las métricas de evaluación nos ayudan a entender cómo se desempeña un modelo en términos de precisión, sensibilidad, capacidad de discriminación, y más.

A continuación, se explican las métricas más importantes y cuándo es útil utilizarlas.

---

## 1. **Accuracy (Precisión)**

La precisión mide el porcentaje de predicciones correctas del modelo en relación con el total de predicciones.

$$
\text{Accuracy} = \frac{\text{True Positives (TP)} + \text{True Negatives (TN)}}{\text{Total de Muestras}}
$$

### ¿Cuándo usarla?
- Es útil cuando las clases están **balanceadas**.
- No es recomendable en casos de **clases desbalanceadas**, ya que una alta precisión puede ser engañosa si el modelo siempre predice la clase mayoritaria.

---

## 2. **Precision (Precisión Positiva)**

Mide la proporción de verdaderos positivos entre todas las predicciones positivas realizadas por el modelo.

$$
\text{Precision} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Positives (FP)}}
$$

### ¿Cuándo usarla?
- Es importante cuando el costo de un **falso positivo** es alto.
- Ejemplo: En un sistema de detección de fraude, se prefiere evitar acusar a usuarios legítimos.
---

## 3. **Recall (Sensibilidad o TPR)**

Mide la proporción de verdaderos positivos que el modelo identifica correctamente.

$$
\text{Recall} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Negatives (FN)}}
$$

### ¿Cuándo usarla?
- Es útil cuando el costo de un **falso negativo** es alto.
- Ejemplo: En problemas médicos, como la detección de enfermedades graves, es crítico identificar todos los casos positivos, incluso a costa de tener más falsos positivos.

---

## 4. **F1-Score**

Es la media armónica de la precisión y la sensibilidad. Combina ambas métricas en un solo valor.

$$
\text{F1-Score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

### ¿Cuándo usarla?
- Es útil en problemas donde hay un **desbalance de clases** y necesitas un equilibrio entre falsos positivos y falsos negativos.

---

## 5. **ROC-AUC (Receiver Operating Characteristic - Area Under the Curve)**

Mide la capacidad del modelo para distinguir entre clases. Se basa en la relación entre:
- **TPR (True Positive Rate o Sensibilidad)**.
- **FPR (False Positive Rate)**.

El AUC (Área Bajo la Curva) varía entre 0 y 1:
- **1.0**: Modelo perfecto.
- **0.5**: Modelo aleatorio.

### ¿Cuándo usarla?
- Es ideal cuando quieres evaluar la **capacidad de discriminación** del modelo, sin importar un umbral de decisión específico.
- Es especialmente útil en problemas de clasificación binaria.

---

## 6. **Precision-Recall Curve (PR-AUC)**

Es otra métrica basada en una curva, pero se enfoca en la relación entre precisión y sensibilidad a diferentes umbrales.

### ¿Cuándo usarla?
- Es más útil que la ROC-AUC cuando las clases están **muy desbalanceadas**.
- Ejemplo: En detección de eventos raros, como fraudes o fallos de máquinas.

---

## 7. **Confusion Matrix (Matriz de Confusión)**

Es una tabla que muestra las predicciones correctas e incorrectas del modelo, divididas por clase:
- **True Positives (TP)**: Predicciones positivas correctas.
- **False Positives (FP)**: Predicciones positivas incorrectas.
- **True Negatives (TN)**: Predicciones negativas correctas.
- **False Negatives (FN)**: Predicciones negativas incorrectas.

### ¿Cuándo usarla?
- Es útil para entender los tipos de errores que comete el modelo.
- Proporciona un análisis detallado, especialmente en problemas multiclase.

---

## 8. **Log Loss (Logarithmic Loss o Entropía Cruzada)**

Mide la calidad de las probabilidades predichas por el modelo, penalizando predicciones incorrectas con alta confianza.

$$
\text{Log Loss} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{j=1}^{M} y_{ij} \log(p_{ij})
$$

Donde $y_{ij}$ es 1 si la clase verdadera es $j$, y $p_{ij}$ es la probabilidad predicha para la clase $j$.

### ¿Cuándo usarla?
- Es útil cuando el modelo produce probabilidades y deseas evaluar la calidad de esas predicciones.
- Ejemplo: En sistemas de recomendación o clasificación probabilística.

---

## Resumen de Uso por Contexto

| **Métrica**          | **Útil Cuando...**                                           | **Ejemplo**                                 |
|-----------------------|-------------------------------------------------------------|---------------------------------------------|
| **Accuracy**          | Clases balanceadas                                         | Clasificación general en tareas balanceadas.|
| **Precision**         | Falsos positivos son costosos                              | Detección de fraude o spam.                 |
| **Recall**            | Falsos negativos son costosos                              | Diagnóstico médico.                         |
| **F1-Score**          | Balance entre precisión y sensibilidad                     | Desbalance de clases en clasificación.      |
| **ROC-AUC**           | Evaluar capacidad de discriminación                        | Modelos binarios en general.                |
| **PR-AUC**            | Clases desbalanceadas                                      | Detección de fallos o eventos raros.        |
| **Confusion Matrix**  | Analizar errores específicos                               | Comprender errores multiclase.              |
| **Log Loss**          | Evaluar probabilidades                                     | Clasificación con salida probabilística.    |

---

Ahora vamos entonces a decidir cuál es el mejor umbral para hacer la distinción entre positivos y negativos. Como métrica usaremos el f1-score:

In [ ]:
# Find the optimal threshold
thresholds = np.linspace(0, 1, 101)
f1_scores = []
for threshold in thresholds:
    y_pred = (y_pred_prob >= threshold).astype(int)
    f1 = f1_score(Y_test, y_pred)
    f1_scores.append(f1)

optimal_threshold = thresholds[np.argmax(f1_scores)]
print(f"Optimal threshold for F1 score: {optimal_threshold}")

Una vez hemos decidido cuál es el umbral óptimo, podemos calcular la matriz de confusión del modelo:

In [ ]:
# Assuming y_pred_prob and Y_test are already defined from previous code
y_pred = (y_pred_prob >= optimal_threshold).astype(int)

# Compute the confusion matrix
cm = confusion_matrix(Y_test, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title(f'Confusion Matrix (Optimal Threshold = {optimal_threshold:.2f})')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

Ahora podemos calcular todas las métricas relevantes y compararlas con los valores base de cada una de ellas:

In [ ]:
# Extract values from the confusion matrix
TN, FP, FN, TP = cm.ravel()

# Calculate metrics
accuracy = accuracy_score(Y_test, y_pred)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = f1_score(Y_test, y_pred)

# Calculate baseline metrics
# Baseline accuracy: predict the majority class
baseline_accuracy = max(Y_test.mean(), 1 - Y_test.mean())

# Baseline precision: the baseline for precision is the prevalence of positives
baseline_precision = Y_test.mean()

# Baseline recall: predicting all positives always yields a recall = 1
baseline_recall = 1

#Baseline F1: Predicting all positives yields the F1 score baseline
baseline_f1 = f1_score(Y_test, [1] * len(Y_test))



print("Confusion Matrix:")
print(cm)
print(f"Accuracy: {accuracy:.4f} (Baseline: {baseline_accuracy:.4f})")
print(f"Precision: {precision:.4f} (Baseline: {baseline_precision:.4f})")
print(f"Recall: {recall:.4f} (Baseline: {baseline_recall:.4f})")
print(f"F1 Score: {f1:.4f} (Baseline: {baseline_f1:.4f})")

#Classification report for more metrics
print(classification_report(Y_test, y_pred))

### Optimización de Hiperparámetros con GridSearch y Validación Cruzada

La optimización de hiperparámetros es un paso crucial en el entrenamiento de modelos de Machine Learning. Los **hiperparámetros** son configuraciones del modelo que no se aprenden automáticamente a partir de los datos, sino que deben ser definidos por el usuario antes del entrenamiento. Algunos ejemplos incluyen la profundidad máxima de un árbol de decisión o el número de vecinos en k-NN.

Dos técnicas clave utilizadas en este proceso son **GridSearch** y **validación cruzada**.

---

#### 1. ¿Qué es GridSearch?

**GridSearch** es una técnica para buscar sistemáticamente la mejor combinación de hiperparámetros en un modelo. Funciona evaluando todas las combinaciones posibles de un conjunto predefinido de valores para los hiperparámetros.

#### Proceso:
1. **Definición de la cuadrícula de búsqueda**:
   - Especificamos un rango de valores para cada hiperparámetro que queremos optimizar.
   - Por ejemplo, para un modelo de SVM:
     ```python
     param_grid = {
         'C': [0.1, 1, 10],
         'kernel': ['linear', 'rbf'],
         'gamma': [0.01, 0.1, 1]
     }
     ```
     Esto define una cuadrícula de 12 combinaciones posibles ($3 \times 2 \times 2$).

2. **Entrenamiento y evaluación**:
   - Para cada combinación de hiperparámetros, el modelo se entrena y evalúa usando un conjunto de validación (generalmente con validación cruzada).

3. **Selección del mejor modelo**:
   - La combinación de hiperparámetros que produce el mejor desempeño en la métrica seleccionada (por ejemplo, F1-score, precisión) se elige como la mejor.

---

#### 2. ¿Qué es la Validación Cruzada?

La **validación cruzada** es una técnica que divide los datos en múltiples particiones (o "folds") para evaluar el desempeño de un modelo de manera más robusta. Esto reduce el riesgo de que los resultados estén sesgados por una única división de los datos.

#### Proceso:
1. Dividir los datos en $k$ particiones (o *folds*).
2. Para cada iteración:
   - Entrenar el modelo en $k-1$ particiones.
   - Evaluar el modelo en la partición restante.
3. Repetir el proceso $k$ veces, rotando el conjunto de validación.
4. Calcular el desempeño promedio del modelo a través de las $k$ particiones.

---

#### 3. Combinando GridSearch y Validación Cruzada

La combinación de GridSearch y validación cruzada asegura que:
- **GridSearch** encuentra la mejor combinación de hiperparámetros.
- **Validación cruzada** garantiza que la evaluación sea robusta y no dependa de una sola partición de datos.

___
Optimicemos los parámetros de la regresión logística validando sobre diferentes posibles valores de los parámetros:
- `C`: define la intensidad de la regularización, entre mas pequeño, más fuerte la regularización.
- `penalty`: define si la regularización es tipo `l1` o `l2`
- `solver`: define el algoritmo a utilizar para minimizar la función de costo.
- `max_iter`: limita el número de iteraciones del algoritmo.

In [ ]:
# Define the parameter grid for Logistic Regression
param_grid = {
    'C': [0.001,0.005, .01, 0.05, 0.1],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [1000]
}

# Create a Logistic Regression model
logreg = LogisticRegression()

# Create a GridSearchCV object
grid_search = GridSearchCV(estimator=logreg, param_grid=param_grid, scoring='roc_auc', cv=5, n_jobs=-1)

# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, Y_train)

# Get the best parameters and best score
best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("Best Parameters:", best_params)
print("Best ROC AUC Score:", best_score)

# Evaluate the best model on the test set
best_model = grid_search.best_estimator_
y_pred_prob = best_model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(Y_test, y_pred_prob)
fpr, tpr, thresholds = roc_curve(Y_test, y_pred_prob)

print(f"ROC AUC on test set: {roc_auc}")

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()

### Regularización en Regresión Logística

La regularización en **regresión logística** es una técnica utilizada para prevenir el sobreajuste (overfitting) y mejorar la capacidad de generalización del modelo. Introduce una penalización sobre los coeficientes ($ \beta $) en la función de costo, limitando la complejidad del modelo al evitar que los coeficientes crezcan demasiado grandes.

---

#### 1. ¿Qué es la regularización?

En la regresión logística, el objetivo es minimizar la función de pérdida basada en la **entropía cruzada**:

$$
J(\beta) = - \frac{1}{N} \sum_{i=1}^N \left[ y_i \log(h_\beta(x_i)) + (1 - y_i) \log(1 - h_\beta(x_i)) \right]
$$

Donde:
- $ y_i $: Clase verdadera ($ 0 $ o $ 1 $).
- $ h_\beta(x_i) $: Probabilidad predicha por el modelo, calculada con la función sigmoide.

La regularización añade un término adicional $ \lambda R(\beta) $ que penaliza los coeficientes $ \beta $, donde:
- $ \lambda $: Controla la fuerza de la penalización (también conocido como hiperparámetro de regularización).
- $ R(\beta) $: Función de penalización (puede ser $ L1 $ o $ L2 $).

La nueva función de costo con regularización es:

$$
J(\beta) = - \frac{1}{N} \sum_{i=1}^N \left[ y_i \log(h_\beta(x_i)) + (1 - y_i) \log(1 - h_\beta(x_i)) \right] + \lambda R(\beta)
$$

---

#### 2. Tipos de Regularización

#### a) **Regularización $ L2 $ (Ridge Regularization)**

La regularización $ L2 $ penaliza la suma de los cuadrados de los coeficientes:

$$
R(\beta) = ||\beta||_2^2 = \sum_{j=1}^m \beta_j^2
$$

Esto significa que el modelo intenta minimizar tanto el error de predicción como la magnitud de los coeficientes.

##### Propiedades:
- **Reducción de coeficientes**: Reduce el valor de los coeficientes grandes pero rara vez los convierte en cero.
- **Uso**: Es útil cuando todas las características son importantes y quieres evitar un modelo sobreajustado.

---

#### b) **Regularización $ L1 $ (Lasso Regularization)**

La regularización $ L1 $ penaliza la suma de los valores absolutos de los coeficientes:

$$
R(\beta) = ||\beta||_1 = \sum_{j=1}^m |\beta_j|
$$

Este tipo de regularización promueve la **sparsidad**, lo que significa que algunos coeficientes se reducen exactamente a cero.

##### Propiedades:
- **Selección de características**: Identifica automáticamente características irrelevantes al asignarles coeficientes cero.
- **Uso**: Es útil cuando sospechas que muchas características no son relevantes y quieres simplificar el modelo.

---

#### 3. Comparación entre $ L1 $ y $ L2 $

| Característica                  | Regularización $ L1 $                  | Regularización $ L2 $                  |
|---------------------------------|------------------------------------------|------------------------------------------|
| **Penalización**                | $ ||\beta||_1 = \sum |\beta_j| $       | $ ||\beta||_2^2 = \sum \beta_j^2 $     |
| **Efecto en los coeficientes**  | Promueve sparsidad (coeficientes en cero)| Reduce coeficientes grandes pero no los anula|
| **Aplicación**                  | Selección de características             | Evitar sobreajuste                       |
| **Uso típico**                  | Modelos interpretables                   | Modelos de alta dimensionalidad          |

---

#### 4. Regularización $ Elastic Net $

El **Elastic Net** combina $ L1 $ y $ L2 $ en una sola fórmula:

$$
R(\beta) = \alpha ||\beta||_1 + (1-\alpha) ||\beta||_2^2
$$

Donde $ \alpha $ controla el peso relativo de $ L1 $ y $ L2 $.

#### Uso:
- Ideal para problemas con muchas características correlacionadas.
- Combina las propiedades de selección de $ L1 $ con la estabilidad de $ L2 $.

---

Veamos las distribuciones del score que arroja el modelo para positivos y negativos:

In [ ]:
# Predict probabilities for the test set using the best model
y_pred_prob = best_model.predict_proba(X_test)[:, 1]

# Separate probabilities for positive and negative cases
positive_probs = y_pred_prob[Y_test == 1]
negative_probs = y_pred_prob[Y_test == 0]

# Define the number of bins
num_bins = 50

# Plot the distributions
plt.figure(figsize=(10, 6))
plt.hist(positive_probs, bins=num_bins, alpha=0.5, label='Positive Cases', density=True)
plt.hist(negative_probs, bins=num_bins, alpha=0.5, label='Negative Cases', density=True)
plt.xlabel('Predicted Probability')
plt.ylabel('Density')
plt.title('Distribution of Predicted Probabilities for Positive and Negative Cases')
plt.legend()
plt.show()

Como se observa, de nuevo el modelo arroja dos distribuciones muy diferentes para positivos y negativos, lo que muestra su poder para discernir los dos tipos de casos. Encontremos de nuevo el mejor valor para el Threshold en este caso:

In [ ]:
# Find the optimal threshold
thresholds = np.linspace(0, 1, 100)
f1_scores = []
for threshold in thresholds:
    y_pred = (y_pred_prob >= threshold).astype(int)
    f1 = f1_score(Y_test, y_pred)
    f1_scores.append(f1)

optimal_threshold = thresholds[np.argmax(f1_scores)]
print(f"Optimal threshold for F1 score: {optimal_threshold}")

# calculate the predictions at optimal threshold
y_pred = (y_pred_prob >= optimal_threshold).astype(int)

# Compute the confusion matrix
cm = confusion_matrix(Y_test, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title(f'Confusion Matrix (Optimal Threshold = {optimal_threshold:.2f})')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

### Pasando de un modelo predictivo, a un modelo de riesgo

Otra posible aplicación de los modelos de clasificación es la estimación del riesgo de ser positivo. Para esto, lo que haremos será hacer una estadística sobre el número de casos positivos para diferentes rangos del score que arroja el modelo.

In [ ]:
# defining the step
d_score = 0.1
y_bins = np.round((y_pred_prob//d_score)*d_score+d_score/2,3)

risk_df = pd.DataFrame(columns = ['score', 'bin_score'])
risk_df['score'] = y_pred_prob
risk_df['bin_score'] = y_bins
risk_df['real'] = Y_test.values
risk_df.head(5)

Hemos creado una base de datos en la que hemos agrupado los datos en el conjunto de prueba en cajas de ancho 0.1. Ahora, para cada caja haremos el cálculo del porcentaje de casos positivos de churn hay.

In [ ]:
risk_data = risk_df.groupby('bin_score')['real'].mean()

Una vez hacemos este cálculo, podemos proceder a graficar

In [ ]:
# use sns barplot to plot bar plot
# between days and tip value
ax = sns.barplot(x=risk_data.index, y=risk_data)

# now simply assign the bar values to
# each bar by passing containers method
# to bar_label function
ax.bar_label(ax.containers[0])
ax.set_xlabel('logistic regression score')
ax.set_ylabel('churn probability')

Como se observa, el modelo ahora permite no sólo hacer predicciones de positivos o negativos, sino también asignar una probabilidad de churn dependiendo del rango en el que el score del modelo caiga. Por ejemplo, para una persona con un score entre 0.4 y 0.5, barra 0.45 en la figura, la probabilidad de hacer churn es del 50%.

### Estimando cuáles son los parámetros más importantes del modelo

Para estimar los parámetros más importantes del modelo, usaremos en este caso un esquema de validación cruzada. Haremos una subdivisión aleatoria en 5 grupos, y ajustaremos el modelo 5 veces, escogiendo un grupo cada vez como conjunto de prueba mientras el resto serán el entrenamiento. Sin embargo, además de esto, haremos este proceso 10 veces. Así, tenemos un total de 50 ajustes diferentes usando diferentes conjuntos entrenamiento-prueba. Sobre todos estos entrenamientos podemos hacer una estadística del valor de los coeficientes y así estimar los parámetros más importantes:

In [ ]:
# Configuración de la validación cruzada repetida
n_splits = 5  # Número de folds
n_repeats = 10  # Número de repeticiones
cv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42)

# Modelo base de regresión logística
model = LogisticRegression(C= 0.1, max_iter= 1000, penalty = 'l1', solver = 'liblinear')

# Almacenar coeficientes de cada repetición
coef_samples = []

# Realizar validación cruzada repetida
for train_idx, test_idx in cv.split(X_train, Y_train):
    # Dividir los datos en entrenamiento y validación
    X_cv_train, y_cv_train = X_train.iloc[train_idx], Y_train.iloc[train_idx]

    # Ajustar el modelo
    model.fit(X_cv_train, y_cv_train)

    # Almacenar los coeficientes
    coef_samples.append(model.coef_[0])  # model.coef_ devuelve una matriz (1, n_features)

# Convertir los coeficientes a un DataFrame
coef_samples = np.array(coef_samples)
coef_df = pd.DataFrame(coef_samples, columns=X_train.columns)

Una vez tenemos el listado de valores de coeficientes, podemos proceder a ver el resultado gráficamente

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.boxplot(data=coef_df)
ax.hlines(y=0, xmin=-1, xmax=len(X_train.columns), color='red', linestyles='dashed')
plt.setp(ax.get_xticklabels(), rotation=90)

La figura muestra la consistencia de resultados entre todos los esquemas de validación cruzada. La distribución de coeficientes es muy pequeña, lo que nos muestra la estabilidad del modelo.

# Gradient Boosting para Clasificación

El **Gradient Boosting** es un poderoso algoritmo de aprendizaje supervisado utilizado para problemas de clasificación y regresión. Funciona mediante la combinación secuencial de varios modelos débiles (como árboles de decisión) para crear un modelo fuerte que minimice el error de predicción.

---

## 1. ¿Cómo funciona Gradient Boosting?

Gradient Boosting utiliza una estrategia aditiva en la que cada modelo débil corrige los errores cometidos por el modelo anterior. Este proceso se basa en la minimización de una función de pérdida mediante un enfoque de descenso por gradiente.

### Pasos principales:
1. **Inicialización**:
   - Se comienza con un modelo base $ F_0(x) $ que predice el valor promedio o una constante (en clasificación, el log-odds inicial).
   - Para la clasificación binaria, se calcula la probabilidad inicial:
     $$
     F_0(x) = \log\left(\frac{\hat{p}_1}{1 - \hat{p}_1}\right)
     $$
     donde $ \hat{p}_1 $ es la proporción de la clase positiva.

2. **Adición secuencial de modelos**:
   - En cada iteración $ m $, se entrena un nuevo modelo débil $ h_m(x) $ sobre el gradiente de la función de pérdida con respecto a las predicciones actuales $ F_{m-1}(x) $.
   - La actualización del modelo es:
     $$
     F_m(x) = F_{m-1}(x) + \nu \cdot h_m(x)
     $$
     donde $ \nu $ es el parámetro de tasa de aprendizaje.

3. **Predicción final**:
   - La predicción se realiza sumando todas las contribuciones de los modelos débiles:
     $$
     F_M(x) = \sum_{m=1}^M \nu \cdot h_m(x)
     $$
   - En clasificación binaria, las probabilidades se obtienen mediante la función sigmoide:
     $$
     \hat{p}(x) = \frac{1}{1 + e^{-F_M(x)}}
     $$

---

In [ ]:
# Crear un modelo base de Gradient Boosting
gb_model = GradientBoostingClassifier(random_state=42)


# Ejecutar la búsqueda de hiperparámetros
gb_model.fit(X_train, Y_train)


Veamos los resultados del modelo

In [ ]:
# Evaluate the best model on the test set
y_pred_prob = gb_model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(Y_test, y_pred_prob)
fpr, tpr, thresholds = roc_curve(Y_test, y_pred_prob)

print(f"ROC AUC on test set: {roc_auc}")

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Gradient Boosting (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()

In [ ]:
itive_probs = y_pred_prob[Y_test == 1]
negative_probs = y_pred_prob[Y_test == 0]

# Define the number of bins
num_bins = 50

# Plot the distributions
plt.figure(figsize=(10, 6))
plt.hist(positive_probs, bins=num_bins, alpha=0.5, label='Positive Cases', density=True)
plt.hist(negative_probs, bins=num_bins, alpha=0.5, label='Negative Cases', density=True)
plt.xlabel('Predicted Probability')
plt.ylabel('Density')
plt.title('Distribution of Predicted Probabilities for Positive and Negative Cases')
plt.legend()
plt.show()

In [ ]:
# Find the optimal threshold
thresholds = np.linspace(0, 1, 100)
f1_scores = []
for threshold in thresholds:
    y_pred = (y_pred_prob >= threshold).astype(int)
    f1 = f1_score(Y_test, y_pred)
    f1_scores.append(f1)

optimal_threshold = thresholds[np.argmax(f1_scores)]
print(f"Optimal threshold for F1 score: {optimal_threshold}")

# calculate the predictions at optimal threshold
y_pred = (y_pred_prob >= optimal_threshold).astype(int)

# Compute the confusion matrix
cm = confusion_matrix(Y_test, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title(f'Confusion Matrix (Optimal Threshold = {optimal_threshold:.2f})')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
# Extract values from the confusion matrix
TN, FP, FN, TP = cm.ravel()

# Calculate metrics
accuracy = accuracy_score(Y_test, y_pred)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = f1_score(Y_test, y_pred)

# Calculate baseline metrics
# Baseline accuracy: predict the majority class
baseline_accuracy = max(Y_test.mean(), 1 - Y_test.mean())

# Baseline precision: the baseline for precision is the prevalence of positives
baseline_precision = Y_test.mean()

# Baseline recall: predicting all positives always yields a recall = 1
baseline_recall = 1

#Baseline F1: Predicting all positives yields the F1 score baseline
baseline_f1 = f1_score(Y_test, [1] * len(Y_test))



print("Confusion Matrix:")
print(cm)
print(f"Accuracy: {accuracy:.4f} (Baseline: {baseline_accuracy:.4f})")
print(f"Precision: {precision:.4f} (Baseline: {baseline_precision:.4f})")
print(f"Recall: {recall:.4f} (Baseline: {baseline_recall:.4f})")
print(f"F1 Score: {f1:.4f} (Baseline: {baseline_f1:.4f})")

#Classification report for more metrics
print(classification_report(Y_test, y_pred))

Como se observa, en este caso el modelo de Gradient Boosting un mayor poder predictivo que el algoritmo de Regresión Logística. 

Finalmente veamos cuales son las características que más impacto tienen en el modelo de clasificación

In [ ]:
feature_importance = gb_model.feature_importances_

# Create a DataFrame for better visualization
feature_importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': feature_importance})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.title('Feature Importance (Gradient Boosting)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

# Máquinas de Soporte Vectorial (SVM)

Las **Máquinas de Soporte Vectorial (SVM)** son algoritmos supervisados utilizados tanto para problemas de clasificación como de regresión. Sin embargo, son más conocidos por su desempeño en tareas de clasificación binaria.

---

## 1. ¿Qué es una SVM?

El objetivo principal de una SVM es encontrar un **hiperplano óptimo** que separe las clases en el espacio de características. Este hiperplano debe maximizar el **margen**, que es la distancia entre el hiperplano y los puntos más cercanos de cada clase (llamados **vectores de soporte**).

### Propiedades principales:
1. **Máximo margen**: Busca la separación más amplia posible entre las clases para mejorar la capacidad de generalización.
2. **Uso de kernels**: Permite mapear datos no linealmente separables a un espacio de características más alto, donde puedan separarse linealmente.

---

## 2. ¿Cómo funciona una SVM?

### a) Separación lineal:
Para datos que son linealmente separables, SVM encuentra un hiperplano $ w \cdot x + b = 0 $ tal que:

$$
w \cdot x + b \geq 1 \quad \text{para puntos positivos}
$$
$$
w \cdot x + b \leq -1 \quad \text{para puntos negativos}
$$

El margen es entonces:
$$
\text{Margen} = \frac{2}{||w||}
$$

El objetivo es maximizar este margen.

---

### b) Datos no linealmente separables:
Para datos complejos, las SVM utilizan **kernels** que transforman los datos a un espacio de características de mayor dimensión. Los kernels más comunes son:
- **Lineal**: Para datos linealmente separables.
- **Polinomial**: Para relaciones no lineales.
- **RBF (Radial Basis Function)**: Basado en funciones gaussianas, ideal para separaciones complejas.
- **Sigmoide**: Similar a las funciones de activación en redes neuronales.

---

### c) Tolerancia al error:
Cuando los datos tienen ruido o solapamiento entre clases, se introduce el parámetro **C**, que controla la compensación entre el margen ancho y la clasificación correcta de los puntos.

- Un **C alto** prioriza la clasificación correcta de todos los puntos, reduciendo el margen.
- Un **C bajo** permite márgenes más amplios a costa de algunos errores de clasificación.

---

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

# Generar datos de ejemplo
X, y = make_blobs(n_samples=100, centers=2, random_state=6)

# Entrenar un modelo SVM
model = SVC(kernel='linear', C=1)
model.fit(X, y)

# Función para graficar el hiperplano y el margen
def plot_svm_decision_boundary(model, X, y):
    plt.scatter(X[:, 0], X[:, 1], c=y, s=30, cmap=plt.cm.Paired)

    # Obtener el hiperplano
    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xx = np.linspace(xlim[0], xlim[1], 30)
    yy = np.linspace(ylim[0], ylim[1], 30)
    YY, XX = np.meshgrid(yy, xx)
    xy = np.vstack([XX.ravel(), YY.ravel()]).T
    Z = model.decision_function(xy).reshape(XX.shape)

    # Dibujar el hiperplano
    ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1], alpha=0.5, linestyles=['--', '-', '--'])
    ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1], s=100, linewidth=1, facecolors='none', edgecolors='k')
    plt.title('SVM con Hiperplano y Vectores de Soporte')
    plt.show()

# Graficar el resultado
plot_svm_decision_boundary(model, X, y)

Apliquemos el SVM a nuestra base de datos. Sin embargo, este modelo puede tomar mucho tiempo de entrenamiento, así que lo aplicaremos sobre un subconjunto de los datos

In [ ]:
# crear un subconjunto aleatorio de datos para X_train y Y_train
X_train_subset, _, Y_train_subset, _ = train_test_split(X_train, Y_train, test_size=0.95, random_state=42, stratify=Y_train)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# Crear pipeline con escalado, PCA y SVM
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2)),  # Puedes ajustar el número de componentes
    ('svm', SVC(kernel='rbf', probability=True, C=1))
])

# Entrenar
pipeline.fit(X_train_subset, Y_train_subset)

In [ ]:
# Evaluar calculando el area bajo la curva ROC
y_pred_prob = pipeline.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(Y_test, y_pred_prob)
print(f"ROC AUC on test set: {roc_auc}")

# Plot the ROC curve
fpr, tpr, thresholds = roc_curve(Y_test, y_pred_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'SVM with PCA (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')


Es posible que con otras configuraciones, el modelo funcione mejor, pero infortunadamente para una base tan grande tomaría mucho tiempo.

## Conclusión

Este caso permitió aplicar modelos de clasificación supervisada como Regresión Logística, SVM y Gradient Boosting para predecir la fuga de clientes, utilizando un preprocesamiento adecuado y métricas como la curva ROC, la matriz de confusión y el F1 Score para evaluar el rendimiento. La combinación de estas técnicas permitió identificar con mayor precisión a los clientes con alta probabilidad de abandono, brindando una herramienta útil para diseñar estrategias preventivas y mejorar la retención. Este enfoque demuestra el valor del aprendizaje automático en la toma de decisiones orientadas al cliente.

### Origen de los datos

Base de datos clientes https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset